# WP5 — Final Composite Hybrid Ranking & Scientific Report

Integrates final structural confidence signals (ESMFold pLDDT and Rg compactness) with the pre-fold composite, 
executes final LLM stability prioritization with structural evidence, validates ranking correlation against deterministic 
evolutionary priors, and generates a scientific interpretative report for top-K candidates.

In [1]:
import sys, pathlib
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
from asr_poc.config import load_config
from asr_poc.io_utils import set_seed
cfg = load_config(ROOT / "config" / "target.yaml")
set_seed(cfg.run.seed)
cfg.paths.ensure_dirs()
print("Target:", cfg.target.name, "| embeddings:", cfg.embeddings.provider, "| report.top_n:", cfg.report.top_n)

Target: lipase | embeddings: local | report.top_n: 3


## 1. Compose final ranking from WP3 signals + WP4 structures

In [2]:
import pandas as pd
from asr_poc import ranking, report
signals = ranking.pre_fold_rank(cfg)  # cached; cheap to recompute
metrics = pd.read_csv(cfg.paths.reports_dir / "structure_metrics.csv")
final = ranking.final_rank(signals, metrics, cfg)
cols = ['rank', 'final_score', 'llm_stability_score', 'deterministic_score', 'struct_conf', 'motif', 'conservation', 'uncertainty', 'validation_rho']
final[cols].round(3)

09:20:04 [info     ] anchors_built                  n=25 out=C:\Users\gcv\Downloads\AI_POC 2\AI_POC\project\data\curated_sequences\anchors.fasta stage=wp3.embeddings
09:20:04 [info     ] embeddings_cache_hit           n=348 path=C:\Users\gcv\Downloads\AI_POC 2\AI_POC\project\embeddings\candidates.parquet stage=wp3.embeddings
09:20:04 [info     ] embeddings_cache_hit           n=25 path=C:\Users\gcv\Downloads\AI_POC 2\AI_POC\project\embeddings\anchors.parquet stage=wp3.embeddings
09:20:04 [info     ] anchor_similarity              k=5 max=0.9895489811897278 mean=0.9761144518852234 min=0.9609580039978027 n=348 stage=wp3.similarity
09:20:04 [info     ] motif_scores                   mean=0.9655172413793104 n=348 n_zero=12 stage=wp3.motif
09:20:11 [info     ] conservation_scores            mean=0.06037646314457451 n=348 stage=wp3.motif strong_cols=545
09:20:12 [info     ] feature_table_built            candidates=348 features=12 stage=wp3.feature_table
09:20:12 [info     ] llm_scoring_requ

C:\Users\gcv\Downloads\AI_POC 2\AI_POC\project\.venv\Lib\site-packages\joblib\externals\loky\backend\context.py:131: UserWarning: Could not find the number of physical cores for the following reason:
[WinError 2] The system cannot find the file specified
Returning the number of logical cores instead. You can silence this warning by setting LOKY_MAX_CPU_COUNT to the number of cores you want to use.
  warnings.warn(
  File "C:\Users\gcv\Downloads\AI_POC 2\AI_POC\project\.venv\Lib\site-packages\joblib\externals\loky\backend\context.py", line 247, in _count_physical_cores
    cpu_count_physical = _count_physical_cores_win32()
                         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\gcv\Downloads\AI_POC 2\AI_POC\project\.venv\Lib\site-packages\joblib\externals\loky\backend\context.py", line 299, in _count_physical_cores_win32
    cpu_info = subprocess.run(
               ^^^^^^^^^^^^^^^
  File "C:\Users\gcv\AppData\Local\Programs\Python\Python311\Lib\subprocess.py", line 548,

09:24:47 [info     ] clustered                      n=348 n_clusters=8 stage=wp3.similarity
09:24:47 [info     ] pre_fold_ranked                n=348 stage=wp3.ranking top_llm_score=8.4 top_pre_score=0.7391783942574754
09:24:47 [info     ] anchors_built                  n=25 out=C:\Users\gcv\Downloads\AI_POC 2\AI_POC\project\data\curated_sequences\anchors.fasta stage=wp3.embeddings
09:24:47 [info     ] embeddings_cache_hit           n=348 path=C:\Users\gcv\Downloads\AI_POC 2\AI_POC\project\embeddings\candidates.parquet stage=wp3.embeddings
09:24:47 [info     ] embeddings_cache_hit           n=25 path=C:\Users\gcv\Downloads\AI_POC 2\AI_POC\project\embeddings\anchors.parquet stage=wp3.embeddings
09:24:47 [info     ] anchor_similarity              k=5 max=0.9895489811897278 mean=0.9761144518852234 min=0.9609580039978027 n=348 stage=wp3.similarity
09:24:47 [info     ] motif_scores                   mean=0.9655172413793104 n=348 n_zero=12 stage=wp3.motif
09:24:54 [info     ] conservation_sc

,rank,final_score,llm_stability_score,deterministic_score,struct_conf,motif,conservation,uncertainty,validation_rho
candidate_id,,,,,,,,,
ALT_Node52_alt2,1,0.791,9.5,0.681,0.601,1.0,0.044,0.0,0.5
ALT_Node47_alt3,2,0.765,9.0,0.676,0.599,1.0,0.044,0.0,0.5
ALT_Node4_alt1,3,0.665,7.0,0.680,0.597,1.0,0.090,0.0,0.5


## 2. Generate the scientific report

In [3]:
import os
print("GEMINI_API_KEY set:", bool(os.getenv("GEMINI_API_KEY")))
path = report.write_report(cfg, final)
print("Wrote:", path)
print(open(path).read())

ANTHROPIC_API_KEY set: False
09:25:21 [info     ] report_written                 n=3 out=C:\Users\gcv\Downloads\AI_POC 2\AI_POC\project\reports\scientific_report.md stage=wp5.report
Wrote: C:\Users\gcv\Downloads\AI_POC 2\AI_POC\project\reports\scientific_report.md
# Scientific Report — Top 3 Candidates (lipase)

Generated: 2026-05-29T09:25:21.359022+00:00

All numbers are computed locally (ESM-2 embeddings + ESMFold + motif/conservation checks). See `reports/candidate_signals.csv` for the full audit trail.

## #1 — `ALT_Node52_alt2`

**Final score:** 0.7913  ·  length 1761 aa  ·  cluster 0

| Signal | Value |
|---|---|
| Anchor cosine (emb_sim) | 0.9888 |
| ESMFold pLDDT/100 (struct_conf) | 0.6005 |
| Catalytic motif | 1.0 |
| Conservation match | 0.044 |
| ASR uncertainty penalty | 0.0 |
| Packing vs benchmarks | 0.3984 |

**Nearest extant lipases:** P06858 (cos=0.9801), O46647 (cos=0.9795), P07867 (cos=0.9793)

**Interpretation:** ESMFold confidence is moderate; the catalytic motif i